# `aidp-streaming-kafka` live test — SASL/PLAIN with OCI auth token
**Live-test row 13.** Mirrors the official Oracle AIDP sample at `oracle-samples/oracle-aidp-samples` (`StreamingFromOCIStreamingService.ipynb`). 1-hour token TTL — refresh before long jobs.


In [ ]:
import sys, os
# Adjust this if you've uploaded the plugin scripts/ dir elsewhere.
sys.path.insert(0, '/Workspace/Shared/oracle_ai_data_platform_connectors/scripts')


In [ ]:
from oracle_ai_data_platform_connectors.streaming import (
    bootstrap_for_region, build_kafka_options_sasl_plain, validate_checkpoint_path,
)

bootstrap = bootstrap_for_region(os.environ['OCI_REGION'])
opts = build_kafka_options_sasl_plain(
    bootstrap_servers=bootstrap,
    tenancy_name=os.environ['OCI_TENANCY_NAME'],
    username=os.environ['OCI_USERNAME'],     # 'oracleidentitycloudservice/<email>' for IAM-Domains
    stream_pool_ocid=os.environ['OCI_STREAM_POOL_OCID'],
    auth_token=os.environ['OCI_AUTH_TOKEN'],
    topic=os.environ['KAFKA_TOPIC'],
    starting_offsets='earliest',
    max_partition_fetch_bytes=1024*1024,
    max_offsets_per_trigger=5,
)


In [ ]:
checkpoint = validate_checkpoint_path(os.environ['KAFKA_CHECKPOINT_VOLUME'])
raw = spark.readStream.format('kafka').options(**opts).load()
out_df = raw.selectExpr("CAST(key AS STRING) AS k", "CAST(value AS STRING) AS v", "topic", "partition", "offset")
query = out_df.writeStream.format('memory').queryName('kafka_apikey_test').option('checkpointLocation', checkpoint).trigger(processingTime='5 seconds').start()
query.awaitTermination(timeout=60)
df = spark.sql('SELECT * FROM kafka_apikey_test')
df.show()
query.stop()
print('input rows in last batch:', (query.lastProgress or {}).get('numInputRows'))


In [ ]:
# Live-test driver parses this final cell's stdout for the JSON summary.
import json, time
summary = {
    'connector': 'aidp-streaming-kafka',
    'auth': 'sasl-plain',
    'rows': int(df.count()),
    'schema': sorted([f.name for f in df.schema.fields]),
    'timestamp_utc': int(time.time()),
}
print('AIDP_LIVE_TEST_RESULT_BEGIN')
print(json.dumps(summary, indent=2))
print('AIDP_LIVE_TEST_RESULT_END')
